# Week 7 — Neural Language Models: Practical Tasks

**Student:** Immaculate Mutheu Muli  
**Reg No:** BSSCS/2024/33678  
**Unit:** BIT4133 Natural Language Processing with Deep Learning

This notebook covers Practical Task 1 (Text Prediction System) and Practical Task 3 (TensorFlow NLP Exercise). Practical Task 2 (the 2-4 page written report) is provided as a separate Word document.

## Setup

In [ ]:
!pip install tensorflow pdfplumber --quiet

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.utils import to_categorical
import numpy as np
import re

print('TensorFlow version:', tf.__version__)
print('Libraries imported successfully.')

## Upload the CBK Annual Report PDF

In [ ]:
from google.colab import files
import pdfplumber

uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]

text_data = ''
with pdfplumber.open(pdf_filename) as pdf:
    for page in pdf.pages[:60]:
        page_text = page.extract_text()
        if page_text:
            text_data += page_text + ' '

text_data = text_data.lower()
text_data = re.sub(r'[^a-z\s.]', ' ', text_data)
text_data = re.sub(r'\s+', ' ', text_data).strip()

print(f'Extracted and cleaned {len(text_data)} characters from the CBK report.')

## Practical Task 1 — Text Prediction System

**Requirements:**
- Create a small dataset (using CBK report sentences)
- Tokenize text
- Train a neural network
- Predict the next word

### Step 1 — Build the Dataset

In [ ]:
sentences = re.split(r'(?<=[.])\s+', text_data)
sentences = [s.strip() for s in sentences if 5 <= len(s.split()) <= 20]

dataset_sentences = sentences[:200]

print(f'Dataset built from {len(dataset_sentences)} CBK report sentences.')
print()
print('Sample sentences:')
for s in dataset_sentences[:3]:
    print(f'  - {s}')

### Step 2 — Tokenize the Text

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(dataset_sentences)

vocab_size = len(tokenizer.word_index) + 1

print(f'Vocabulary size: {vocab_size} unique words')
print()
print('Sample of word index:')
sample_items = list(tokenizer.word_index.items())[:10]
for word, idx in sample_items:
    print(f'  {word}: {idx}')

### Step 3 — Create Input Sequences for Next-Word Prediction

For every sentence, we create multiple training examples by progressively building up the sequence. For example the sentence "inflation declined sharply this year" produces:

```
[inflation] -> declined
[inflation, declined] -> sharply
[inflation, declined, sharply] -> this
[inflation, declined, sharply, this] -> year
```

In [ ]:
input_sequences = []

for sentence in dataset_sentences:
    token_list = tokenizer.texts_to_sequences([sentence])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

max_seq_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

X = input_sequences[:, :-1]
y = input_sequences[:, -1]
y = to_categorical(y, num_classes=vocab_size)

print(f'Total training sequences created: {len(input_sequences)}')
print(f'Max sequence length: {max_seq_len}')
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

### Step 4 — Build and Train the Neural Network

This network uses an Embedding layer (turns word numbers into dense vectors), an LSTM layer (learns sequence patterns), and a Dense output layer with Softmax (predicts probability for every possible next word).

In [ ]:
model = Sequential([
    Embedding(vocab_size, 32, input_length=max_seq_len - 1),
    LSTM(64),
    Dense(vocab_size, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()

In [ ]:
history = model.fit(X, y, epochs=50, verbose=1)

print()
print(f'Final training accuracy: {history.history["accuracy"][-1]:.4f}')
print(f'Final training loss:     {history.history["loss"][-1]:.4f}')

### Step 5 — Predict the Next Word

Type a phrase from CBK-style language and see what word the model predicts next.

In [ ]:
def predict_next_word(seed_text, model, tokenizer, max_seq_len):
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')
    predicted_probs = model.predict(token_list, verbose=0)[0]
    predicted_index = np.argmax(predicted_probs)
    confidence = predicted_probs[predicted_index]

    predicted_word = ''
    for word, index in tokenizer.word_index.items():
        if index == predicted_index:
            predicted_word = word
            break

    return predicted_word, confidence

test_phrases = [
    'the central bank',
    'inflation declined',
    'monetary policy',
    'the kenya shilling'
]

print('NEXT-WORD PREDICTIONS')
print('=' * 50)
for phrase in test_phrases:
    word, conf = predict_next_word(phrase, model, tokenizer, max_seq_len)
    print(f'"{phrase}" -> predicted next word: "{word}"  (confidence: {conf:.2%})')

## Practical Task 3 — TensorFlow NLP Exercise

**Requirements:**
- Accept text input
- Tokenize the text
- Display word indices
- Convert text into sequences

In [ ]:
user_text = input('Type a sentence about the CBK report: ')

exercise_tokenizer = Tokenizer()
exercise_tokenizer.fit_on_texts([user_text])

print()
print('YOUR TEXT:')
print(user_text)
print()
print('WORD INDICES:')
print(exercise_tokenizer.word_index)
print()
print('TEXT CONVERTED TO SEQUENCE:')
print(exercise_tokenizer.texts_to_sequences([user_text]))

## Summary

This notebook covered:
- Building a complete next-word prediction system trained on CBK report sentences
- Using Embedding + LSTM + Dense(softmax) layers
- Training the model and observing accuracy improve over epochs
- Testing the trained model with new phrases and confidence scores
- A TensorFlow exercise accepting live text input, tokenizing it, and converting it to sequences